# Downloading SRTM GL1 Elevation

Source: [USGS/SRTMGL1_003](https://developers.google.com/earth-engine/datasets/catalog/USGS_SRTMGL1_003). Native 30m (1 arc-second), near-global coverage (60°N–56°S). Single band `elevation` in meters.

Export scale = **100m** to match the Hansen `treecover2000` / `loss` downloads. For a continuous field like elevation, GEE's default pyramid policy is `mean`, so 100m pixels are the average elevation of the underlying ~11×11 sub-pixel block.

Exports go to Google Drive via Earth Engine. Monitor at the [GEE Task Manager](https://code.earthengine.google.com/tasks).

In [2]:
import ee
import geemap

ee.Authenticate()  # run once if not already authenticated
ee.Initialize()

In [3]:
# SRTM covers 60°N to 56°S (no data at the poles)
srtm_bbox = ee.Geometry.BBox(-180, -56, 180, 60)

elevation = ee.Image("USGS/SRTMGL1_003").select("elevation")

print(elevation.getInfo()["id"])  # sanity-check

USGS/SRTMGL1_003


In [4]:
# Quick preview
vis_params = {
    "min": 0,
    "max": 4000,
    "palette": ["000080", "0000ff", "00ffff", "00ff00", "ffff00", "ff8000", "ff0000", "ffffff"],
}

Map = geemap.Map(center=[0, 0], zoom=2)
Map.addLayer(elevation.clip(srtm_bbox), vis_params, "SRTM GL1 Elevation (m)")
Map.addLayer(srtm_bbox, {}, "Region")
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [5]:
# Export to Google Drive at 100m (downsampled from native 30m)
task = ee.batch.Export.image.toDrive(
    image=elevation,
    description="srtm_elevation_100m_30m",
    folder="GEE_exports",
    fileNamePrefix="srtm_elevation_100m_30m",
    region=srtm_bbox,
    scale=100,
    crs="EPSG:4326",
    maxPixels=1e13,
)

task.start()
print("Export task started:", task.id)

Export task started: 2PODN3I7R4GF4HXABLVE5ZZW


### NOTE: track the export at the [GEE Task Manager](https://code.earthengine.google.com/tasks)

Suggest placing output tiles in `maps/raw/SRTM/` in the Dropbox project folder, alongside the other raster inputs.